# Kaggle — House Prices : Advanced Regression Techniques

In [2]:
import pandas as pd
import seaborn as sns
import numpy as np

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
test_ids = test["Id"]  # Id du test, nécessaires pour la soumission

train.shape, test.shape

((1460, 81), (1459, 80))

## 1. Préparation des données

In [3]:
train = train.reset_index(drop=True)

# Cible passée en log (la métrique Kaggle est un RMSE sur le log du prix)
y = np.log1p(train["SalePrice"])

# Features uniquement : on retire l'identifiant et la cible
X_train = train.drop(columns=["Id", "SalePrice"])
X_test  = test.drop(columns=["Id"])

# Concaténation train + test ; n_train sert à re-séparer plus tard
n_train = len(X_train)
all_X = pd.concat([X_train, X_test], axis=0, ignore_index=True)

In [4]:
all_X.shape  # attendu : (2919, 79)

(2919, 79)

## 2. Valeurs manquantes

In [5]:
# Colonnes qui contiennent au moins un NaN
cols_na = all_X.columns[all_X.isnull().any()]
df_null = all_X[cols_na]

# Nombre et pourcentage de NaN, triés par ordre décroissant
na = df_null.isnull().sum().sort_values(ascending=False)
na_pct = (na / len(all_X) * 100).round(1)
pd.DataFrame({"nb_NaN": na, "pct": na_pct})

,nb_NaN,pct
PoolQC,2909,99.7
MiscFeature,2814,96.4
Alley,2721,93.2
Fence,2348,80.4
MasVnrType,1766,60.5
FireplaceQu,1420,48.6
LotFrontage,486,16.6
GarageQual,159,5.4
GarageYrBlt,159,5.4
GarageCond,159,5.4


### 2.1 `PoolQC` — qualité de la piscine (99,7 % de NaN)

D'après `data_description.txt`, `NA` = pas de piscine. On vérifie que les NaN correspondent aux maisons avec `PoolArea == 0`.

In [6]:
# Seulement 10 piscines ont une qualité renseignée
all_X['PoolQC'].value_counts()

PoolQC
Ex    4
Gd    4
Fa    2
Name: count, dtype: int64

**Test de cohérence `PoolQC` × `PoolArea`**

In [7]:
# Maisons sans piscine
(all_X['PoolArea'] == 0).sum()

np.int64(2906)

In [8]:
# Pas de piscine ET pas de qualité -> cohérent
((all_X['PoolArea'] == 0) & (all_X['PoolQC'].isnull())).sum()

np.int64(2906)

In [9]:
# Piscine présente MAIS qualité manquante -> vraies valeurs manquantes
((all_X['PoolArea'] > 0) & (all_X['PoolQC'].isnull())).sum()

np.int64(3)

In [10]:
# Pas de piscine MAIS qualité renseignée -> incohérence
((all_X['PoolArea'] == 0) & (all_X['PoolQC'].notnull())).sum()

np.int64(0)

In [11]:
# Piscine présente ET qualité renseignée -> cohérent
((all_X['PoolArea'] > 0) & (all_X['PoolQC'].notnull())).sum()

np.int64(10)

**Résultat**

| | `PoolQC` vide | `PoolQC` renseigné |
|---|---|---|
| `PoolArea == 0` | 2 906 | 0 |
| `PoolArea > 0` | 3 | 10 |

- 2 906 NaN = pas de piscine.
- 3 piscines sans qualité = vraies valeurs manquantes.
- Aucune incohérence dans l'autre sens.

In [12]:
# Les 3 piscines SANS qualité : train ou test ?
mask = (all_X["PoolArea"] > 0) & (all_X["PoolQC"].isnull())
idx = all_X[mask].index

print(idx)
print(["train" if i < n_train else "test" for i in idx])

Index([2420, 2503, 2599], dtype='int64')
['test', 'test', 'test']


In [13]:
# Les 10 piscines AVEC qualité : train ou test ?
mask = (all_X['PoolArea'] > 0) & (all_X['PoolQC'].notnull())
idx = all_X[mask].index

print(idx)
print(["train" if i < n_train else "test" for i in idx])

Index([197, 810, 1170, 1182, 1298, 1386, 1423, 1974, 2573, 2710], dtype='int64')
['train', 'train', 'train', 'train', 'train', 'train', 'train', 'test', 'test', 'test']


**Conclusion**

- Les 3 piscines sans qualité sont toutes dans le test.
- Seulement 7 piscines notées dans le train sur 1 460 maisons : trop peu pour que le modèle apprenne l'effet de la qualité.
- `PoolArea` contient déjà l'information sur la présence de piscine.

**Décision : suppression de `PoolQC`.**

In [14]:
# Suppression de PoolQC (trop peu d'exemples notés dans le train)
all_X = all_X.drop(columns=["PoolQC"])
all_X.shape  # attendu : (2919, 78)

(2919, 78)

### 2.2 `MiscFeature` — équipement divers (96,4 % de NaN)

D'après `data_description.txt`, `NA` = aucun équipement, et `MiscVal` = valeur en $ de cet équipement. On vérifie que les NaN correspondent aux maisons avec `MiscVal == 0`.

In [15]:
# Répartition des équipements renseignés
all_X['MiscFeature'].value_counts()

MiscFeature
Shed    95
Gar2     5
Othr     4
TenC     1
Name: count, dtype: int64

In [16]:
# Nombre total d'équipements renseignés
all_X['MiscFeature'].value_counts().sum()

np.int64(105)

**Test de cohérence `MiscFeature` × `MiscVal`**

In [17]:
# Maisons sans valeur d'équipement
(all_X['MiscVal'] == 0).sum()

np.int64(2816)

In [18]:
# Pas de valeur ET pas d'équipement -> cohérent
((all_X['MiscVal'] == 0) & (all_X['MiscFeature'].isnull())).sum()

np.int64(2813)

In [19]:
# Valeur renseignée MAIS équipement manquant -> incohérence
((all_X['MiscVal'] > 0) & (all_X['MiscFeature'].isnull())).sum()

np.int64(1)

In [20]:
# Équipement présent MAIS valeur à 0
((all_X['MiscVal'] == 0) & (all_X['MiscFeature'].notnull())).sum()

np.int64(3)

In [21]:
# Équipement présent ET valeur renseignée -> cohérent
((all_X['MiscVal'] > 0) & (all_X['MiscFeature'].notnull())).sum()

np.int64(102)

**Résultat**

| | `MiscFeature` vide | `MiscFeature` renseigné |
|---|---|---|
| `MiscVal == 0` | 2 813 | 3 |
| `MiscVal > 0` | 1 | 102 |

- 2 813 NaN = aucun équipement.
- 1 maison avec une valeur mais sans équipement, 3 équipements avec une valeur à 0.

In [22]:
# Répartition des équipements par catégorie, entre train et test
condition = (all_X['MiscVal'] > 0) & (all_X['MiscFeature'].notnull())
sub = all_X.loc[condition, ['MiscFeature']].copy()
sub['origine'] = ["train" if i < n_train else "test" for i in sub.index]
pd.crosstab(sub['MiscFeature'], sub['origine'], margins=True)

origine,test,train,All
MiscFeature,,,
Gar2,3,2,5
Othr,2,1,3
Shed,45,48,93
TenC,0,1,1
All,50,52,102


**Conclusion**

- Répartition train / test d'environ 50/50 (52 / 50).
- Dans le train, seule `Shed` a assez d'exemples (48) ; `Gar2`, `Othr` et `TenC` en ont 1 ou 2 : le modèle ne peut rien en apprendre.
- `MiscVal` porte déjà l'information utile (présence d'un équipement et sa valeur) ; la catégorie apporte très peu en plus.
- Garder `MiscFeature` imposerait un encodage pour un gain quasi nul.

**Décision : suppression de `MiscFeature`, on conserve `MiscVal`.**

In [23]:
# Suppression de MiscFeature (MiscVal conserve l'information utile)
all_X = all_X.drop(columns=["MiscFeature"])
all_X.shape  # attendu : (2919, 77)

(2919, 77)

### 2.3 `Alley` — type d'accès par allée (93,2 % de NaN)

D'après `data_description.txt`, `NA` = pas d'accès par une allée. Aucune autre colonne ne décrit l'allée : pas de test de cohérence possible, le NaN se lit directement comme « absent ».

In [24]:
# Répartition des types d'allée renseignés
all_X['Alley'].value_counts()

Alley
Grvl    120
Pave     78
Name: count, dtype: int64

In [25]:
# Nombre total d'allées renseignées
all_X['Alley'].value_counts().sum()

np.int64(198)

In [26]:
# Répartition des types d'allée entre train et test
sub = all_X[["Alley"]].copy()
sub['origine'] = ["train" if i < n_train else "test" for i in sub.index]
pd.crosstab(sub['Alley'], sub['origine'], margins=True)

origine,test,train,All
Alley,,,
Grvl,70,50,120
Pave,37,41,78
All,107,91,198


**Conclusion**

- Effectifs suffisants dans le train : `Grvl` 50, `Pave` 41.
- Pas d'ordre entre gravier et pavé : encodage one-hot (pas ordinal).
- Aucune autre colonne ne porte cette information.

**Décision : NaN remplacés par `"None"`.**
Le one-hot sera fait à la fin, en une seule fois sur toutes les colonnes catégorielles ; la colonne `Alley_None` pourra alors être supprimée (redondante avec `Alley_Grvl = Alley_Pave = 0`).

In [27]:
# NaN = pas d'allée
all_X["Alley"] = all_X["Alley"].fillna("None")

In [28]:
# Vérification : plus aucun NaN dans Alley
all_X['Alley'].value_counts()

Alley
None    2721
Grvl     120
Pave      78
Name: count, dtype: int64

### 2.4 `Fence` — qualité de la clôture (80,4 % de NaN)

D'après `data_description.txt`, `NA` = pas de clôture. Valeurs : `GdPrv` (bonne intimité), `MnPrv` (intimité minimale), `GdWo` (bon bois), `MnWw` (bois/grillage minimal). Aucune autre colonne ne décrit la clôture : le NaN se lit directement comme « absent ».

In [29]:
# Répartition des types de clôture renseignés
all_X['Fence'].value_counts()

Fence
MnPrv    329
GdPrv    118
GdWo     112
MnWw      12
Name: count, dtype: int64

In [30]:
# Nombre total de clôtures renseignées
all_X['Fence'].value_counts().sum()

np.int64(571)

In [31]:
# Répartition des types de clôture entre train et test
sub = all_X[["Fence"]].copy()
sub['origine'] = ["train" if i < n_train else "test" for i in sub.index]
pd.crosstab(sub['Fence'], sub['origine'], margins=True)

origine,test,train,All
Fence,,,
GdPrv,59,59,118
GdWo,58,54,112
MnPrv,172,157,329
MnWw,1,11,12
All,290,281,571


**Conclusion**

- Répartition train / test d'environ 50/50 (281 / 290).
- Effectifs suffisants dans le train sauf `MnWw` (11).
- Deux familles de clôture (intimité / bois), chacune avec un niveau minimum et un niveau bon.

**Décision : NaN remplacés par `"None"`.**
Encodage à l'étape finale. Piste à comparer en CV avec un one-hot classique : deux colonnes ordinales `Fence_Privacy` (None 0, MnPrv 1, GdPrv 2) et `Fence_Wood` (None 0, MnWw 1, GdWo 2), possibles car chaque maison a au plus une clôture.

In [32]:
# NaN = pas de clôture, puis vérification
all_X["Fence"] = all_X["Fence"].fillna("None")
all_X['Fence'].value_counts()

Fence
None     2348
MnPrv     329
GdPrv     118
GdWo      112
MnWw       12
Name: count, dtype: int64

### 2.5 `MasVnrType` — type de placage maçonnerie (60,5 % de NaN)

D'après `data_description.txt`, `MasVnrType` n'a pas de valeur `NA` mais une valeur `None` (pas de placage) : pandas convertit automatiquement le texte `"None"` en NaN à la lecture. `MasVnrArea` (surface du placage) a aussi 23 NaN. On croise les deux colonnes.

In [33]:
# Répartition des types de placage renseignés
all_X['MasVnrType'].value_counts()

MasVnrType
BrkFace    879
Stone      249
BrkCmn      25
Name: count, dtype: int64

In [34]:
# Croise 'colonne catégorielle vide / renseignée' avec 'colonne numérique NaN / 0 / > 0'
# (réutilisable pour les groupes Garage*, Bsmt*, Fireplace*...)
def coherence(df, col_cat, col_num):
    etat_cat = df[col_cat].isnull().map({True: "vide", False: "renseigné"}).rename(col_cat)
    etat_num = pd.Series(
        np.select([df[col_num].isnull(), df[col_num] == 0], ["NaN", "0"], default="> 0"),
        index=df.index, name=col_num
    )
    return pd.crosstab(etat_cat, etat_num, margins=True)

In [35]:
# Test de cohérence MasVnrType × MasVnrArea
coherence(all_X, "MasVnrType", "MasVnrArea")

MasVnrArea,0,> 0,NaN,All
MasVnrType,,,,
renseigné,3,1150,0,1153
vide,1735,8,23,1766
All,1738,1158,23,2919


In [36]:
# Répartition des types de placage entre train et test
sub = all_X[["MasVnrType"]].copy()
sub['origine'] = ["train" if i < n_train else "test" for i in sub.index]
pd.crosstab(sub['MasVnrType'], sub['origine'], margins=True)

origine,test,train,All
MasVnrType,,,
BrkCmn,10,15,25
BrkFace,434,445,879
Stone,121,128,249
All,565,588,1153


**Conclusion**

- 1 735 maisons sans type et sans surface : pas de placage.
- 23 maisons sans type ni surface : vraies valeurs manquantes sur les deux colonnes ; la majorité des maisons n'ayant pas de placage, on les considère sans placage.
- 8 maisons avec une surface mais sans type : la surface prouve qu'il y a un placage, on leur attribue le type le plus fréquent du train (`BrkFace`).
- 3 maisons avec un type mais une surface à 0 : laissées telles quelles (pas de NaN, impossible de savoir quelle colonne est fausse).
- Pas d'ordre entre les types ; `BrkCmn` a 15 exemples dans le train, suffisant pour le garder.

**Décision** : `BrkFace` pour les 8 maisons, puis `MasVnrType` → `"None"` et `MasVnrArea` → 0 pour le reste. One-hot à l'étape finale.

In [37]:
# Les 8 maisons avec une surface mais sans type -> type le plus fréquent du train
mode_type = all_X.iloc[:n_train]["MasVnrType"].mode()[0]
condition = (all_X['MasVnrArea'] > 0) & (all_X['MasVnrType'].isnull())
all_X.loc[condition, ['MasVnrType']] = mode_type

In [38]:
# Reste des NaN = pas de placage
all_X["MasVnrType"] = all_X["MasVnrType"].fillna("None")

In [39]:
# Surface manquante = pas de placage
all_X["MasVnrArea"] = all_X["MasVnrArea"].fillna(0)

In [40]:
# Vérification : None uniquement avec une surface à 0, plus aucun NaN
etat_area = np.where(all_X["MasVnrArea"] > 0, "> 0", "0")
pd.crosstab(all_X["MasVnrType"], etat_area, margins=True, colnames=["MasVnrArea"])

MasVnrArea,0,> 0,All
MasVnrType,,,
BrkCmn,0,25,25
BrkFace,2,885,887
None,1758,0,1758
Stone,1,248,249
All,1761,1158,2919


### 2.6 `FireplaceQu` — qualité de la cheminée (48,6 % de NaN)

D'après `data_description.txt`, `NA` = pas de cheminée. Échelle de qualité : `Po` < `Fa` < `TA` < `Gd` < `Ex`. On vérifie la cohérence avec `Fireplaces` (nombre de cheminées).

In [41]:
# Répartition des niveaux de qualité renseignés
all_X['FireplaceQu'].value_counts()

FireplaceQu
Gd    744
TA    592
Fa     74
Po     46
Ex     43
Name: count, dtype: int64

In [42]:
# Nombre total de cheminées avec une qualité renseignée
all_X['FireplaceQu'].value_counts().sum()

np.int64(1499)

In [43]:
# Test de cohérence FireplaceQu × Fireplaces
coherence(all_X, "FireplaceQu", "Fireplaces")

Fireplaces,0,> 0,All
FireplaceQu,,,
renseigné,0,1499,1499
vide,1420,0,1420
All,1420,1499,2919


**Conclusion**

- Cohérence parfaite : les 1 420 NaN correspondent exactement aux maisons sans cheminée, les 1 499 qualités renseignées aux maisons avec au moins une cheminée.
- Échelle de qualité ordonnée : encodage **ordinal** avec le dictionnaire commun des qualités (`None` 0, `Po` 1, `Fa` 2, `TA` 3, `Gd` 4, `Ex` 5). Les niveaux rares (`Po`, `Ex`) ne posent pas de problème en ordinal.

**Décision : NaN remplacés par `"None"`.** Encodage ordinal à l'étape finale.

In [44]:
# NaN = pas de cheminée, puis vérification
all_X["FireplaceQu"] = all_X["FireplaceQu"].fillna("None")
all_X['FireplaceQu'].value_counts()

### 2.7 `LotFrontage` — longueur de façade sur rue (16,6 % de NaN)